# Confidence Interval Analysis

Reads experiment log CSVs from a chosen experiment directory and computes
mean F1, standard deviation, and 95% confidence intervals.  
Configure the paths in the **Configuration** cell below.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

ROOT = Path("..").resolve()

# ── Configuration ──────────────────────────────────────────────────────────────
# Choose one experiment directory to analyse:
EXPERIMENT_DIR = ROOT / "experiments-finetuning"
# EXPERIMENT_DIR = ROOT / "experiments-not-finetuning"
# EXPERIMENT_DIR = ROOT / "experiments-twitter"
# EXPERIMENT_DIR = ROOT / "experiments-swin"
# EXPERIMENT_DIR = ROOT / "exps-tuning-handle-subjectivity"

# Filter: only include experiment directories whose name starts with this prefix.
# Set to "" to include all experiments in the directory.
PREFIX_FILTER = "deepseek-llama3"
# ──────────────────────────────────────────────────────────────────────────────

data_paths = sorted(
    str(EXPERIMENT_DIR / d / "logs" / "test_logs.csv")
    for d in os.listdir(EXPERIMENT_DIR)
    if d.startswith(PREFIX_FILTER) and
       (EXPERIMENT_DIR / d / "logs" / "test_logs.csv").exists()
)
print(f"Experiment dir : {EXPERIMENT_DIR}")
print(f"Filter prefix  : '{PREFIX_FILTER}' → {len(data_paths)} experiment(s) found")

## F1-score confidence intervals

In [ ]:
CONFIDENCE = 0.95

for data_path in data_paths:
    df = pd.read_csv(data_path)
    f1_scores = df["f1_score"].to_numpy()
    mean_f1   = np.mean(f1_scores)
    dof       = len(f1_scores) - 1
    ci        = stats.t.interval(CONFIDENCE, dof, loc=mean_f1, scale=stats.sem(f1_scores))

    exp_name = Path(data_path).parts[-3]
    print(f"\n{'─'*55}")
    print(f"  {exp_name}")
    print(f"{'─'*55}")
    print(f"  Mean F1  : {mean_f1 * 100:.2f}%")
    print(f"  Std Dev  : {f1_scores.std() * 100:.2f}%")
    print(f"  95% CI   : [{ci[0]*100:.2f}%, {ci[1]*100:.2f}%]")
    if "time" in df.columns:
        total_h = df["time"].sum() / 3600
        print(f"  Total time: {total_h:.2f} h")

## Subjectivity regression experiments (Pearson / MSE / MAE)

In [ ]:
SUBJECTIVITY_DIR = ROOT / "exps-tuning-handle-subjectivity"
SUBJECTIVITY_PREFIX = "openai-modernbert"

reg_paths = sorted(
    str(SUBJECTIVITY_DIR / d / "logs" / "test_logs.csv")
    for d in os.listdir(SUBJECTIVITY_DIR)
    if d.startswith(SUBJECTIVITY_PREFIX) and
       (SUBJECTIVITY_DIR / d / "logs" / "test_logs.csv").exists()
)
print(f"{len(reg_paths)} regression experiment(s) found\n")

for data_path in reg_paths:
    df = pd.read_csv(data_path)
    pearsons = df["val_pearson"].to_numpy()
    mses     = df["val_mse"].to_numpy()
    maes     = df["val_mae"].to_numpy()
    mean_p   = np.mean(pearsons)
    dof      = len(pearsons) - 1
    ci_p     = stats.t.interval(CONFIDENCE, dof, loc=mean_p, scale=stats.sem(pearsons))

    exp_name = Path(data_path).parts[-3]
    print(f"\n{'─'*55}")
    print(f"  {exp_name}")
    print(f"{'─'*55}")
    print(f"  Pearson  : {mean_p:.4f}  (std {np.std(pearsons, ddof=1):.4f})")
    print(f"  95% CI   : [{ci_p[0]:.4f}, {ci_p[1]:.4f}]")
    print(f"  MSE      : {np.mean(mses):.4f}  (std {np.std(mses, ddof=1):.4f})")
    print(f"  MAE      : {np.mean(maes):.4f}  (std {np.std(maes, ddof=1):.4f})")
    if "time_sec" in df.columns:
        print(f"  Time     : {df['time_sec'].sum() / 3600:.2f} h")